In [3]:
"""
Standard Brute-Force Gaussian Process Regression on SARCOS Dataset
Baseline for comparison with Kronecker-structured GP.

WARNING: This implementation has O(N^3) time and O(N^2) memory complexity.
Using the full dataset (44k samples) will likely cause OOM on standard machines.
We default to a subset of the data.
"""

import numpy as np
import scipy.io
import sys
import time
from scipy.spatial.distance import cdist
from scipy.linalg import cholesky, solve_triangular
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

class BruteForceGP:
    """
    Standard Gaussian Process Regression.
    Complexity: O(N^3) time, O(N^2) memory.
    """
    
    def __init__(self, sigma_f=1.0, sigma_n=0.1, length_scale=1.0):
        self.sigma_f = sigma_f
        self.sigma_n = sigma_n
        self.length_scale = length_scale
        self.X_train = None
        self.L = None
        self.alpha = None
        self.memory_usage = {}
        
    def _kernel(self, X1, X2):
        """Standard RBF Kernel: k(x, x') = σ_f² * exp(-||x-x'||² / 2l²)"""
        # Efficient distance computation
        dists = cdist(X1, X2, 'sqeuclidean')
        return self.sigma_f**2 * np.exp(-dists / (2 * self.length_scale**2))
        
    def fit(self, X, y):
        """Fit the GP model using Cholesky decomposition."""
        n = X.shape[0]
        self.X_train = X
        y = y.reshape(-1, 1)
        
        print(f"Fitting Brute Force GP on {n} samples...")
        sys.stdout.flush()
        t0 = time.time()
        
        # 1. Compute Kernel Matrix (O(N^2))
        print(f"  Computing Kernel Matrix ({n}x{n})...")
        sys.stdout.flush()
        K = self._kernel(X, X)
        
        # Memory check
        mem_K = K.nbytes / (1024**2)
        print(f"  Kernel matrix memory: {mem_K:.2f} MB")
        self.memory_usage['kernel_mb'] = mem_K
        
        # 2. Add noise to diagonal
        K[np.diag_indices_from(K)] += self.sigma_n**2
        
        # 3. Cholesky Decomposition (O(N^3))
        print("  Computing Cholesky Decomposition (in-place to save memory)...")
        sys.stdout.flush()
        try:
            # overwrite_a=True destroys K but saves memory (replaces K with L)
            # We don't need K again for prediction (we compute K_* separately)
            self.L = cholesky(K, lower=True, overwrite_a=True)
        except np.linalg.LinAlgError:
            print("  Cholesky failed! Matrix might not be positive definite. Adding jitter...")
            # If failed, K might be partially destroyed, so we might need to recompute or handle carefully.
            # However, for this script, we'll try to recover if possible, but real recovery implies recomputing K.
            # Since we can't easily undo overwrite_a, we'll raise for now or warn.
            raise RuntimeError("Cholesky failed with overwrite_a=True. Rerun with higher sigma_n.")
            
        # 4. Solve for alpha (O(N^2))
        # alpha = K^{-1} y = L^{-T} L^{-1} y
        print("  Solving for weights...")
        sys.stdout.flush()
        self.alpha = solve_triangular(self.L.T, solve_triangular(self.L, y, lower=True))
        
        t_fit = time.time() - t0
        print(f"Fitting done in {t_fit:.2f}s")
        self.memory_usage['total_mb'] = mem_K  # Dominant term
        
        return self
        
    def predict(self, X_test, return_std=False):
        """Predict on new data."""
        n_test = X_test.shape[0]
        print(f"Predicting on {n_test} samples...")
        sys.stdout.flush()
        t0 = time.time()
        
        # Compute K_star (covariance between train and test)
        K_star = self._kernel(self.X_train, X_test) # (N_train, N_test)
        
        # Mean prediction: mu = K_star^T @ alpha
        mu = K_star.T @ self.alpha
        
        if return_std:
            # Variance: k(x*, x*) - K_star^T @ K^{-1} @ K_star
            # K^{-1} = L^{-T} L^{-1}
            # v = L^{-1} @ K_star
            # var = k(x*, x*) - v^T @ v
            
            print("  Computing variance...")
            # Compute variance of test points (diagonal of kernel)
            # For RBF, k(x,x) = sigma_f^2
            k_star_diag = np.full(n_test, self.sigma_f**2)
            
            # Compute v = L^{-1} K_star
            v = solve_triangular(self.L, K_star, lower=True)
            
            # Predictive variance
            var = k_star_diag - np.sum(v**2, axis=0) + self.sigma_n**2 # Add noise variance for predictive distribution
            std = np.sqrt(np.maximum(var, 1e-10))
            
            print(f"Prediction done in {time.time() - t0:.2f}s")
            return mu.ravel(), std
            
        print(f"Prediction done in {time.time() - t0:.2f}s")
        return mu.ravel()

def main():
    print("\n" + "="*60)
    print("BRUTE FORCE GP ON SARCOS (BASELINE)")
    print("="*60 + "\n")
    
    # Load data
    try:
        train_data = scipy.io.loadmat('Datasets/sarcos_inv.mat')
        test_data = scipy.io.loadmat('Datasets/sarcos_inv_test.mat')
    except FileNotFoundError:
        print("Error: Dataset files not found in 'Datasets/' directory.")
        return

    train_key = [k for k in train_data.keys() if not k.startswith('__')][0]
    test_key = [k for k in test_data.keys() if not k.startswith('__')][0]
    
    X_train_full = train_data[train_key][:, :21]
    y_train_full = train_data[train_key][:, 21]
    X_test_full = test_data[test_key][:, :21]
    y_test_full = test_data[test_key][:, 21]
    
    # Normalize data
    scaler_X = StandardScaler()
    X_train_s = scaler_X.fit_transform(X_train_full)
    X_test_s = scaler_X.transform(X_test_full)
    
    scaler_y = StandardScaler()
    y_train_s = scaler_y.fit_transform(y_train_full.reshape(-1, 1)).ravel()
    
    # -------------------------------------------------------------
    # CONFIGURATION
    # -------------------------------------------------------------
    # Use all features of the dataset
    # Select all 21 features available in the SARCOS dataset
    feature_indices = list(range(21)) 
    
    # SUBSET SIZE
    # User requested full dataset run on 32GB RAM system.
    # We will use all training data.
    # N ~ 44,484
    # Memory for Kernel Matrix (float64): ~15 GB
    # We use overwrite_a=True in Cholesky to avoid doubling memory usage.
    
    print(f"Using FULL training set: {X_train_full.shape[0]} samples")
    print("WARNING: This will use ~15GB of RAM for the kernel matrix alone.")
    print("Ensure you have at least 32GB RAM available.")
    
    X_train = X_train_s[:, feature_indices]
    y_train = y_train_s
    
    # Use full test set but subset features
    X_test = X_test_s[:, feature_indices]
    y_test_orig = y_test_full
    
    n_test = len(X_test)
    
    print(f"Training Data: {X_train.shape}")
    print(f"Test Data:     {X_test.shape}")
    
    # Hyperparameters (same as Kronecker for rough comparison, though optimal might differ)
    # sigma_f=1.0, sigma_n=0.08, length_scale=1.2
    gp = BruteForceGP(sigma_f=1.0, sigma_n=0.08, length_scale=1.2)
    
    # Train
    print("\n" + "="*60)
    print("TRAINING")
    print("="*60)
    # Hint to users about time
    print("Note: complexity is O(N^3). For N=44k, this may take hours.")
    gp.fit(X_train, y_train)
    
    # Predict
    print("\n" + "="*60)
    print("PREDICTION")
    print("="*60)
    y_pred_s, y_std_s = gp.predict(X_test, return_std=True)
    
    # Inverse transform predictions
    y_pred = scaler_y.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
    
    # Metrics
    print("\n" + "="*60)
    print("RESULTS")
    print("="*60)
    
    mse = mean_squared_error(y_test_orig, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_orig, y_pred)
    smse = mse / np.var(y_test_orig)
    
    print(f"MSE      : {mse:.4f}")
    print(f"RMSE     : {rmse:.4f}")
    print(f"R²       : {r2:.4f}")
    print(f"SMSE     : {smse:.4f}")
    
    print("\nPerformance:")
    print(f"Kernel Memory: {gp.memory_usage.get('kernel_mb', 0):.2f} MB")
    
    # Plotting
    try:
        print("\nPlotting results...")
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # Pred vs True
        ax = axes[0]
        # Downsample for scatter plot if too many points
        plot_idx = np.random.choice(n_test, min(2000, n_test), replace=False)
        ax.scatter(y_test_orig[plot_idx], y_pred[plot_idx], alpha=0.5, s=10)
        lims = [min(y_test_orig.min(), y_pred.min()), max(y_test_orig.max(), y_pred.max())]
        ax.plot(lims, lims, 'r--', alpha=0.8)
        ax.set_xlabel('True Values')
        ax.set_ylabel('Predicted Values')
        ax.set_title(f'Brute Force GP (Full Data)\nR² = {r2:.3f}')
        ax.grid(True, alpha=0.3)
        
        # Residuals
        ax = axes[1]
        residuals = y_test_orig - y_pred
        ax.hist(residuals, bins=50, edgecolor='k', alpha=0.7)
        ax.set_xlabel('Residual')
        ax.set_ylabel('Count')
        ax.set_title('Residual Distribution')
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig('sarcos_gp_bruteforce_results.png', dpi=150)
        print("Saved plot to 'sarcos_gp_bruteforce_results.png'")
        plt.close()
    except Exception as e:
        print(f"Plotting failed: {e}")

    print("\n" + "="*60)
    print("DONE")
    print("="*60)

if __name__ == "__main__":
    main()



BRUTE FORCE GP ON SARCOS (BASELINE)

Using FULL training set: 44484 samples
Ensure you have at least 32GB RAM available.
Training Data: (44484, 21)
Test Data:     (4449, 21)

TRAINING
Note: complexity is O(N^3). For N=44k, this may take hours.
Fitting Brute Force GP on 44484 samples...
  Computing Kernel Matrix (44484x44484)...
  Kernel matrix memory: 15097.25 MB
  Computing Cholesky Decomposition (in-place to save memory)...
  Solving for weights...
Fitting done in 1469.24s

PREDICTION
Predicting on 4449 samples...
  Computing variance...
Prediction done in 52.26s

RESULTS
MSE      : 0.7588
RMSE     : 0.8711
R²       : 0.9982
SMSE     : 0.0018

Performance:
Kernel Memory: 15097.25 MB

Plotting results...
Saved plot to 'sarcos_gp_bruteforce_results.png'

DONE


In [2]:
"""
Optimized Kronecker-Structured GP Regression on SARCOS Dataset

TRUE KRONECKER ACCELERATION with corrected prediction formula.
"""

import numpy as np
import scipy.io
import sys
from scipy.linalg import eigh, cholesky, solve_triangular
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import time


class OptimizedKroneckerGP:
    """
    Optimized GP with TRUE Kronecker acceleration.
    
    Uses FITC-style sparse approximation with Kronecker structure.
    """
    
    def __init__(self, m=5, sigma_f=1.0, sigma_n=0.1, length_scale=1.0):
        self.m = m
        self.sigma_f = sigma_f
        self.sigma_n = sigma_n
        self.length_scale = length_scale
        
        self.D = None
        self.Z = []
        # Kronecker eigendecomposition storage
        self.Q_factors = []  # Eigenvectors of each K_d
        self.lambda_factors = []  # Eigenvalues of each K_d
        self.alpha = None
        
    def _rbf_1d(self, x1, x2):
        """1D RBF kernel"""
        x1 = x1.reshape(-1, 1)
        x2 = x2.reshape(-1, 1)
        dist = cdist(x1, x2, 'sqeuclidean')   
        return np.exp(-dist / (2 * self.length_scale**2))
    
    def _compute_Kuf(self, X):
        """
        Compute K_uf using product kernel structure.
        k(z, x) = σ² ∏_d k_d(z_d, x_d)
        """
        n = X.shape[0]
        M = self.m ** self.D
        
        # Compute factor kernels
        K_factors = []
        for d in range(self.D):
            K_d = self._rbf_1d(self.Z[d], X[:, d])  # (m, n)
            K_factors.append(K_d)
        
        # Build K_uf
        K_uf = self.sigma_f**2 * np.ones((M, n))
        
        for d in range(self.D):
            n_repeat = self.m ** d
            n_tile = self.m ** (self.D - d - 1)
            K_d_exp = np.tile(K_factors[d], (n_tile, 1))
            K_d_exp = np.repeat(K_d_exp, n_repeat, axis=0)
            K_uf *= K_d_exp
        
        return K_uf
    
    def _compute_Kuu_eigen(self):
        """
        Compute eigendecomposition of K_uu factors.
        Instead of forming K_uu = K_1 ⊗ K_2 ⊗ ... ⊗ K_D,
        we eigen-decompose each K_d separately.
        
        This reduces:
        - Time: O(M³) → O(Dm³)
        - Memory: O(M²) → O(Dm²)
        """
        # Compute and eigen-decompose each factor matrix
        Q_factors = []
        lambda_factors = []
        
        for d in range(self.D):
            K_d = self.sigma_f**(2.0/self.D) * self._rbf_1d(self.Z[d], self.Z[d])
            K_d += 1e-6 * np.eye(self.m)
            
            # Eigendecomposition: K_d = Q_d Λ_d Q_d^T
            lambdas, Q = eigh(K_d)
            lambda_factors.append(lambdas)
            Q_factors.append(Q)
        
        return Q_factors, lambda_factors
    
    def _kronecker_eigenvalues(self):
        """
        Compute Kronecker product of eigenvalues.
        Λ = Λ_1 ⊗ Λ_2 ⊗ ... ⊗ Λ_D
        
        Returns: 1D array of length M = m^D
        """
        # Start with first set of eigenvalues
        kron_lambda = self.lambda_factors[0]
        
        # Kronecker product of eigenvalues
        for d in range(1, self.D):
            kron_lambda = np.kron(kron_lambda, self.lambda_factors[d])
        
        return kron_lambda
    
    def _kron_mv(self, X, transpose=False):
        """
        Efficiently compute Q @ X or Q^T @ X where Q = Q_1 ⊗ Q_2 ⊗ ... ⊗ Q_D
        without forming the full Kronecker product.
        
        Args:
            X: vector of length M = m^D or matrix of shape (M, n)
            transpose: if True, compute Q^T @ X instead of Q @ X
        
        Returns:
            Transformed vector/matrix of same shape as input
        """
        M = self.m ** self.D
        
        # Handle both vector and matrix inputs
        is_vector = (X.ndim == 1)
        if is_vector:
            X = X.reshape(-1, 1)
        
        n_cols = X.shape[1]
        result = X.copy()
        
        # Reshape into D-dimensional tensor + columns: (m, m, ..., m, n_cols)
        shape = [self.m] * self.D + [n_cols]
        result = result.reshape(shape)
        
        # Apply each Q_d along its corresponding axis
        for d in range(self.D):
            Q_d = self.Q_factors[d]
            if transpose:
                Q_d = Q_d.T
            
            # Move axis d to position 0, apply Q_d, move back
            result = np.moveaxis(result, d, 0)
            original_shape = result.shape
            result = result.reshape(self.m, -1)
            result = Q_d @ result
            result = result.reshape(original_shape)
            result = np.moveaxis(result, 0, d)
        
        # Reshape back to (M, n_cols)
        result = result.reshape(M, n_cols)
        
        if is_vector:
            return result.ravel()
        return result
    
    def _calculate_memory_usage(self, n, M):
        """
        Calculate memory usage and savings from Kronecker eigenvalue structure.
        
        Optimized Kronecker eigenvalue approach stores:
        - Q_factors: D × (m × m) eigenvector matrices
        - lambda_factors: D × m eigenvalue vectors
        - K_uf: M × n (cross-covariance)
        - Alpha: M × 1
        
        Old Kronecker approach (Cholesky):
        - K_uu: M × M
        - K_uf: M × n
        - L: M × M (Cholesky factor)
        
        Full GP would need:
        - K_nn: n × n (full covariance)
        """
        bytes_per_float = 8  # float64
        
        # Optimized Kronecker eigenvalue memory (actual)
        mem_Q = self.D * self.m * self.m * bytes_per_float  # Eigenvectors
        mem_lambda = self.D * self.m * bytes_per_float  # Eigenvalues
        mem_Kuf = M * n * bytes_per_float
        mem_alpha = M * bytes_per_float
        optimized_memory = mem_Q + mem_lambda + mem_Kuf + mem_alpha
        
        # Old Kronecker (Cholesky) memory
        mem_Kuu_old = M * M * bytes_per_float
        mem_L_old = M * M * bytes_per_float
        old_kronecker_memory = mem_Kuu_old + mem_Kuf + mem_L_old + mem_alpha
        
        # Full GP memory (hypothetical)
        mem_Knn = n * n * bytes_per_float
        full_gp_memory = mem_Knn
        
        # Savings vs old Kronecker
        kron_savings = old_kronecker_memory - optimized_memory
        kron_savings_pct = (kron_savings / old_kronecker_memory) * 100
        
        # Savings vs full GP
        full_savings = full_gp_memory - optimized_memory
        full_savings_pct = (full_savings / full_gp_memory) * 100
        
        return {
            'optimized_mb': optimized_memory / (1024**2),
            'old_kronecker_mb': old_kronecker_memory / (1024**2),
            'full_gp_mb': full_gp_memory / (1024**2),
            'kron_savings_mb': kron_savings / (1024**2),
            'kron_savings_pct': kron_savings_pct,
            'full_savings_mb': full_savings / (1024**2),
            'full_savings_pct': full_savings_pct,
            'n': n,
            'M': M,
            'm': self.m,
            'D': self.D
        }
    
    def fit(self, X, y):
        """
        Fit using sparse GP approximation.
        
        Uses Woodbury identity:
        (K_uf K_fu / σ² + K_uu)⁻¹ 
        """
        n, self.D = X.shape
        M = self.m ** self.D
        y = y.ravel()
        
        print(f"Fitting Optimized Kronecker GP")
        print(f"  Samples: {n}, Dimensions: {self.D}")
        print(f"  Inducing/dim: {self.m}, Total: {M:,}")
        sys.stdout.flush()
        t0 = time.time()
        
        # Create inducing points
        self.Z = []
        for d in range(self.D):
            q = np.linspace(0.05, 0.95, self.m)
            self.Z.append(np.quantile(X[:, d], q))
        
        # Compute eigendecomposition of K_uu factors
        print("Computing Kronecker eigendecomposition...")
        sys.stdout.flush()
        t_eigen = time.time()
        self.Q_factors, self.lambda_factors = self._compute_Kuu_eigen()
        print(f"  Eigendecomposition done in {time.time() - t_eigen:.2f}s")
        sys.stdout.flush()
        
        # Compute K_uf
        print("Computing K_uf...")
        sys.stdout.flush()
        K_uf = self._compute_Kuf(X)  # (M, n)
        
        # Get Kronecker eigenvalues
        kron_lambda = self._kronecker_eigenvalues()  # (M,)
        
        # Sparse GP using eigendecomposition:
        # Σ = K_uu + K_uf K_fu / σ² = Q Λ Q^T + K_uf K_fu / σ²
        print("Solving sparse GP system using eigenvalues...")
        sys.stdout.flush()
        
        # Transform K_uf: K_uf_tilde = Q^T @ K_uf
        print(f"  Transforming K_uf ({M} x {n})...")
        sys.stdout.flush()
        K_uf_tilde = self._kron_mv(K_uf, transpose=True)
        
        # Σ_tilde = Λ + K_uf_tilde @ K_uf_tilde^T / σ²
        # For efficiency, compute via eigenvalues + low-rank update
        # α_tilde = (Λ + K_uf_tilde K_uf_tilde^T / σ²)^{-1} K_uf_tilde y / σ²
        
        # Use Woodbury identity: (Λ + UU^T/σ²)^{-1} = Λ^{-1} - Λ^{-1}U(σ²I + U^TΛ^{-1}U)^{-1}U^TΛ^{-1}
        # For speed, we use conjugate gradient or direct solve on the transformed system
        
        # Direct approach for moderate M:
        Sigma_tilde = np.diag(kron_lambda) + K_uf_tilde @ K_uf_tilde.T / self.sigma_n**2
        Sigma_tilde += 1e-6 * np.eye(M)  # Regularization
        
        # Cholesky on diagonal-dominant matrix (much more stable)
        L_tilde = cholesky(Sigma_tilde, lower=True)
        
        # Solve for alpha_tilde: α_tilde = Σ_tilde^{-1} K_uf_tilde y / σ²
        b_tilde = K_uf_tilde @ y / self.sigma_n**2
        alpha_tilde = solve_triangular(
            L_tilde.T,
            solve_triangular(L_tilde, b_tilde, lower=True)
        )
        
        # Transform back: α = Q @ α_tilde
        self.alpha = self._kron_mv(alpha_tilde, transpose=False)
        
        # Store for prediction
        self.L_tilde = L_tilde
        self.kron_lambda = kron_lambda
        
        # Calculate memory usage
        self.memory_info = self._calculate_memory_usage(n, M)
        
        print(f"Fitting done in {time.time() - t0:.2f}s")
        sys.stdout.flush()
        return self
    
    def predict(self, X_test, return_std=False):
        """Predict using sparse GP"""
        n_test = X_test.shape[0]
        print(f"Predicting on {n_test} samples...")
        sys.stdout.flush()
        t0 = time.time()
        
        K_sf = self._compute_Kuf(X_test)  # (M, n_test)
        
        # Mean: K_*uᵀ α
        y_pred = K_sf.T @ self.alpha
        
        if return_std:
            print("Computing uncertainties...")
            sys.stdout.flush()
            
            # Variance: k(x*,x*) - K_*uᵀ K_uu⁻¹ K_*u + K_*uᵀ Σ⁻¹ K_*u
            k_star = self.sigma_f**2
            
            # Transform K_sf: K_sf_tilde = Q^T @ K_sf
            K_sf_tilde = self._kron_mv(K_sf, transpose=True)
            
            # Solve L_tilde v = K_sf_tilde for variance computation
            v1 = solve_triangular(self.L_tilde, K_sf_tilde, lower=True)
            
            # K_uu⁻¹ term using eigenvalues: K_uu^{-1} = Q Λ^{-1} Q^T
            # v2 = Λ^{-0.5} Q^T K_sf
            lambda_inv_sqrt = 1.0 / np.sqrt(self.kron_lambda + 1e-8)
            v2 = lambda_inv_sqrt[:, None] * K_sf_tilde
            
            y_var = k_star - np.sum(v2**2, axis=0) + np.sum(v1**2, axis=0) + self.sigma_n**2
            y_std = np.sqrt(np.maximum(y_var, 1e-8))
            
            print(f"Prediction done in {time.time() - t0:.2f}s")
            return y_pred, y_std
        
        print(f"Prediction done in {time.time() - t0:.2f}s")
        return y_pred


def main():
    print("\n" + "="*60)
    print("OPTIMIZED KRONECKER GP ON SARCOS")
    print("="*60 + "\n")
    
    # Load data
    train_data = scipy.io.loadmat('Datasets/sarcos_inv.mat')
    test_data = scipy.io.loadmat('Datasets/sarcos_inv_test.mat')
    
    train_key = [k for k in train_data.keys() if not k.startswith('__')][0]
    test_key = [k for k in test_data.keys() if not k.startswith('__')][0]
    
    X_train_full = train_data[train_key][:, :21]
    y_train_full = train_data[train_key][:, 21]
    X_test_full = test_data[test_key][:, :21]
    y_test_full = test_data[test_key][:, 21]
    
    print(f"Full data: {X_train_full.shape[0]} train, {X_test_full.shape[0]} test")
    
    # Normalize
    scaler_X = StandardScaler()
    X_train_s = scaler_X.fit_transform(X_train_full)
    X_test_s = scaler_X.transform(X_test_full)
    
    scaler_y = StandardScaler()
    y_train_s = scaler_y.fit_transform(y_train_full.reshape(-1, 1)).ravel()
    
    # Use full dataset
    D = 5  # Dimensions
    m = 6  # Inducing per dim -> 6^5 = 7776
    
    # Better feature selection: mix of positions and velocities
    feature_indices = [0, 1, 2, 7, 8]  # First 3 positions + first 2 velocities
    
    # Use all data (no subsampling)
    X_train = X_train_s[:, feature_indices]
    y_train = y_train_s
    y_train_orig = y_train_full
    
    X_test = X_test_s[:, feature_indices]
    y_test_orig = y_test_full
    
    n_train = len(X_train)
    n_test = len(X_test)
    
    print(f"\nUsing: {n_train} train, {n_test} test, D={D}, m={m}")
    print(f"Kronecker size: {m}^{D} = {m**D:,}")
    sys.stdout.flush()
    
    # Fit
    print("\n" + "="*60)
    print("TRAINING")
    print("="*60)
    
    gp = OptimizedKroneckerGP(m=m, sigma_f=1.0, sigma_n=0.08, length_scale=1.2)
    gp.fit(X_train, y_train)
    
    # Predict
    print("\n" + "="*60)
    print("PREDICTION")
    print("="*60)
    
    y_pred_s, y_std_s = gp.predict(X_test, return_std=True)
    
    y_pred = scaler_y.inverse_transform(y_pred_s.reshape(-1, 1)).ravel()
    y_std = y_std_s * scaler_y.scale_[0]
    
    # Metrics
    print("\n" + "="*60)
    print("RESULTS")
    print("="*60)
    
    mse = mean_squared_error(y_test_orig, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_orig, y_pred)
    smse = mse / np.var(y_test_orig)
    
    print(f"MSE      : {mse:.4f}")
    print(f"RMSE     : {rmse:.4f}")
    print(f"R²       : {r2:.4f}")
    print(f"SMSE     : {smse:.4f}")
    
    # Memory efficiency
    print("\n" + "="*60)
    print("MEMORY EFFICIENCY (Optimized Kronecker Eigenvalue Structure)")
    print("="*60)
    mem = gp.memory_info
    print(f"\nOptimized (Eigenvalue) : {mem['optimized_mb']:.2f} MB")
    print(f"Old Kronecker (Cholesky) : {mem['old_kronecker_mb']:.2f} MB")
    print(f"Full GP Memory           : {mem['full_gp_mb']:.2f} MB")
    print(f"\nSavings vs Old Kronecker : {mem['kron_savings_mb']:.2f} MB ({mem['kron_savings_pct']:.1f}%)")
    print(f"Savings vs Full GP       : {mem['full_savings_mb']:.2f} MB ({mem['full_savings_pct']:.1f}%)")
    print(f"\nCompression (m^D vs Dm^2): {mem['M']:,} -> {mem['D']}x{mem['m']}^2 = {mem['D'] * mem['m']**2:,}")
    print(f"Factor reduction         : {mem['M'] / (mem['D'] * mem['m']**2):.1f}×")
    
    # Plot
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f'Kronecker GP (m={m}, D={D}, n={n_train})', fontsize=14)
    
    ax = axes[0, 0]
    ax.scatter(y_test_orig, y_pred, alpha=0.5, s=10)
    lims = [min(y_test_orig.min(), y_pred.min()), max(y_test_orig.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--')
    ax.set_xlabel('True'); ax.set_ylabel('Predicted')
    ax.set_title(f'R² = {r2:.3f}'); ax.grid(alpha=0.3)
    
    ax = axes[0, 1]
    residuals = y_test_orig - y_pred
    ax.scatter(y_pred, residuals, alpha=0.5, s=10)
    ax.axhline(0, color='r', linestyle='--')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Residuals')
    ax.set_title('Residuals'); ax.grid(alpha=0.3)
    
    ax = axes[1, 0]
    ax.hist(residuals, bins=40, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='r', linestyle='--')
    ax.set_xlabel('Residuals'); ax.set_ylabel('Count')
    ax.set_title('Residual Distribution'); ax.grid(alpha=0.3)
    
    ax = axes[1, 1]
    idx = np.argsort(y_test_orig)
    step = max(1, n_test // 50)
    i = np.arange(0, n_test, step)
    ax.errorbar(i, y_pred[idx][i], yerr=2*y_std[idx][i], fmt='r.', alpha=0.5, label='Pred±2σ')
    ax.plot(i, y_test_orig[idx][i], 'b.', alpha=0.7, label='True')
    ax.legend(); ax.set_xlabel('Index (sorted)'); ax.set_ylabel('Value')
    ax.set_title('Predictions with Uncertainty'); ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('sarcos_gp_results.png', dpi=200)
    print(f"\nSaved: sarcos_gp_results.png")
    plt.close()
    
    print("\n" + "="*60)
    print("DONE!")
    print("="*60)


if __name__ == "__main__":
    main()


OPTIMIZED KRONECKER GP ON SARCOS

Full data: 44484 train, 4449 test

Using: 44484 train, 4449 test, D=5, m=6
Kronecker size: 6^5 = 7,776

TRAINING
Fitting Optimized Kronecker GP
  Samples: 44484, Dimensions: 5
  Inducing/dim: 6, Total: 7,776
Computing Kronecker eigendecomposition...
  Eigendecomposition done in 0.00s
Computing K_uf...
Solving sparse GP system using eigenvalues...
  Transforming K_uf (7776 x 44484)...
Fitting done in 40.04s

PREDICTION
Predicting on 4449 samples...
Computing uncertainties...
Prediction done in 4.52s

RESULTS
MSE      : 139.9998
RMSE     : 11.8322
R²       : 0.6620
SMSE     : 0.3380

MEMORY EFFICIENCY (Optimized Kronecker Eigenvalue Structure)

Optimized (Eigenvalue) : 2639.13 MB
Old Kronecker (Cholesky) : 3561.77 MB
Full GP Memory           : 15097.25 MB

Savings vs Old Kronecker : 922.64 MB (25.9%)
Savings vs Full GP       : 12458.12 MB (82.5%)

Compression (m^D vs Dm^2): 7,776 -> 5x6^2 = 180
Factor reduction         : 43.2×

Saved: sarcos_gp_results.